# Extração de Amostra de Dados SRAG

## 1. Importações e definição de diretórios

Esta célula funciona tanto se o notebook for executado da **raiz do projeto** quanto de dentro de `notebooks/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


# Raiz do projeto (funciona tanto da raiz quanto de notebooks/)
ROOT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

# Permite importar src
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from src.data.extracao import coletar_amostra, csv_to_parquet

print("ROOT_DIR:", ROOT_DIR)

## 2. Configuração do diretório raiz e diretório de dados brutos

In [ ]:
RAW_DIR = ROOT_DIR / "data" / "raw"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

## 3. Coleta da amostra

⚠️ **Pendência conhecida:** a URL atualmente usada por `coletar_amostra()` em `src/data/extracao.py` está retornando **404 NOT FOUND**:

```
https://apidadosabertos.saude.gov.br/srag?limit=1000&offset=0
```

Antes de executar esta célula, é preciso corrigir a função `baixar_pagina()` / o endpoint padrão em `src/data/extracao.py`.

Enquanto isso, considere o dataset já validado em `00_acesso_dados_parquet.ipynb` (Parquet distribuído via GitHub Release, ~4.445.191 registros) como fonte oficial.

In [ ]:
df = coletar_amostra(n_paginas=50)

print(f"\nTotal coletado: {len(df):,} registros")
print(f"Total de colunas: {len(df.columns)}")

if df.empty:
    raise ValueError("Nenhum registro foi coletado.")

df.head()

## 4. Persistência dos dados em CSV

In [ ]:
arquivo = RAW_DIR / "srag_amostra.csv"

df.to_csv(
    arquivo,
    index=False,
    encoding="utf-8-sig"
)

## 5. Verificação do total de registros salvos

In [ ]:
df_salvo = pd.read_csv(arquivo, low_memory=False)

print(f"Arquivo salvo em: {arquivo}")
print(f"Registros no DataFrame: {len(df):,}")
print(f"Registros no arquivo:   {len(df_salvo):,}")

if len(df) == len(df_salvo):
    print("✓ Verificação concluída: todos os registros foram salvos.")
else:
    print("⚠ Atenção: o número de registros não coincide.")

## 6. Conversão dos dados para Parquet

In [ ]:
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

arquivo_parquet = PROCESSED_DIR / "srag.parquet"

csv_to_parquet(
    csv_path=arquivo,
    out_path=arquivo_parquet
)

## 7. Verificação dos arquivos gerados

In [ ]:
print(f"CSV:     {arquivo}")
print(f"Parquet: {arquivo_parquet}")

print(f"\nTamanho do CSV:     {arquivo.stat().st_size / 1024**2:.2f} MB")
print(f"Tamanho do Parquet: {arquivo_parquet.stat().st_size / 1024**2:.2f} MB")